## step 1

In [ ]:
import pandas as pd
from glob import glob
import os

home_data_list = glob("/nas/houce/UK_mobility/data_20251226/home/home_UK_2020_*.csv")
home_data_list.sort(key=lambda x: os.path.basename(x).split('_')[1] + os.path.basename(x).split('_')[2])

workplace_data_list = glob("/nas/houce/UK_mobility/data_20251226/workplace/workplace_UK_2020_*.csv")
workplace_data_list.sort(key=lambda x: os.path.basename(x).split('_')[1] + os.path.basename(x).split('_')[2])

In [ ]:
home_data = pd.read_csv("/nas/houce/UK_mobility/data_20251226/home/home_UK_2020_0201.csv")
workplace_data = pd.read_csv("/nas/houce/UK_mobility/data_20251226/workplace/workplace_UK_2020_0201.csv")

In [ ]:
from tqdm import tqdm
combined_datas = []
workplace_datas = []
for i, file_path in tqdm(enumerate(home_data_list), total=len(home_data_list)):
    home_df = pd.read_csv(file_path)
    workplace_df = pd.read_csv(workplace_data_list[i])
    home_df['workplace'] = 0
    combined_df = pd.concat([home_df, workplace_df], ignore_index=True)
    combined_datas.append(combined_df)

combined_datas_df = pd.concat(combined_datas, ignore_index=True)

In [ ]:
combined_datas_df = pd.concat(combined_datas, ignore_index=True)
combined_datas_df_filter = combined_datas_df[combined_datas_df['showed_hours_today'] >= 12].reset_index(drop=True)
total_hours = combined_datas_df_filter.groupby(['device_aid','latitude', 'longitude']).agg({'duration_location_hour':'sum'}).reset_index()

In [ ]:
def find_and_save_primary_location(df, id_col, duration_col, location_cols, location_type_name, output_path):
    """
    从给定的DataFrame中根据最长停留时间找出每个ID的主要位置，
    将其保存到CSV，并返回一个带有位置类型标记的DataFrame。

    参数:
        df (pd.DataFrame): 包含设备ID、位置和停留时间的输入DataFrame。
        id_col (str): 设备标识符列的名称。
        duration_col (str): 停留时间列的名称。
        location_cols (list): 定义位置的列名列表 (例如, ['device_aid', 'latitude', 'longitude'])。
        location_type_name (str): 识别出的位置类型的名称 (例如, 'home' 或 'workplace')。
        output_path (str): 保存输出CSV文件的路径。

    返回:
        pd.DataFrame: 一个新的DataFrame，其中包含了标记了新位置类型的原始数据。
    """
    # 按ID分组，找到每组中最大停留时间对应的行索引
    idx = df.groupby(id_col)[duration_col].idxmax()
    
    # 提取主要位置信息
    primary_locations = df.loc[idx, location_cols].reset_index(drop=True)
    
    # 保存主要位置到CSV文件
    primary_locations.to_csv(output_path, index=False)
    print(f"已将主要的 {location_type_name} 位置保存到 {output_path}")
    
    # 创建一个用于合并的临时DataFrame，并添加位置类型标记
    location_flags = primary_locations.copy()
    location_flags[location_type_name] = 1
    
    # 将标记合并回原始的df，以识别所有位置
    # 注意：这里使用原始传入的df进行合并，以标记所有相关行
    df_with_flags = df.merge(location_flags, on=location_cols, how='left')
    df_with_flags[location_type_name].fillna(0, inplace=True)
    
    return df_with_flags, primary_locations


In [ ]:
# --- 使用函数识别并保存家庭位置 ---
# 第一次调用函数，识别家庭位置
total_hours_with_home, home_location_2020 = find_and_save_primary_location(
    df=total_hours,
    id_col='device_aid',
    duration_col='duration_location_hour',
    location_cols=['device_aid', 'latitude', 'longitude'],
    location_type_name='home',
    output_path="/nas/houce/UK_mobility/data_20251226/home_workplace_location/home_location_2020.csv"
)

# --- 准备数据以识别工作场所 ---
# 从已标记家庭位置的数据中，筛选出所有非家庭的位置
total_hours_not_home = total_hours_with_home[total_hours_with_home['home'] == 0].copy()

# --- 使用函数识别并保存工作场所位置 ---
# 第二次调用函数，在非家庭位置中识别工作场所
total_hours_not_home_with_workplace, workplace_location_2020 = find_and_save_primary_location(
    df=total_hours_not_home,
    id_col='device_aid',
    duration_col='duration_location_hour',
    location_cols=['device_aid', 'latitude', 'longitude'],
    location_type_name='workplace',
    output_path="/nas/houce/UK_mobility/data_20251226/home_workplace_location/workplace_location.csv"
)

# --- 合并家庭和工作场所的标记 ---
# 为了得到一个包含家庭和工作场所完整标记的最终DataFrame，我们将工作场所的标记合并回带有家庭标记的DataFrame
final_df = total_hours_with_home.merge(
    total_hours_not_home_with_workplace[['device_aid', 'latitude', 'longitude', 'workplace']],
    on=['device_aid', 'latitude', 'longitude'],
    how='left'
)
final_df['workplace'].fillna(0, inplace=True)

print("\n处理完成。")
print("家庭位置识别结果:")
print(home_location_2020.head())
print("\n工作场所位置识别结果:")
print(workplace_location_2020.head())
print("\n带有家庭和工作场所标记的最终DataFrame:")
print(final_df.head())

In [ ]:
home_data_list = glob("/nas/houce/UK_mobility/data_20251226/home/home_uk_2021_*.csv")
home_data_list.sort(key=lambda x: os.path.basename(x).split('_')[1] + os.path.basename(x).split('_')[2])

workplace_data_list = glob("/nas/houce/UK_mobility/data_20251226/workplace/workplace_uk_2021_*.csv")
workplace_data_list.sort(key=lambda x: os.path.basename(x).split('_')[1] + os.path.basename(x).split('_')[2])


from tqdm import tqdm
combined_datas = []
workplace_datas = []
for i, file_path in tqdm(enumerate(home_data_list), total=len(home_data_list)):
    home_df = pd.read_csv(file_path)
    workplace_df = pd.read_csv(workplace_data_list[i])
    home_df['workplace'] = 0
    combined_df = pd.concat([home_df, workplace_df], ignore_index=True)
    combined_df = combined_df.drop(columns=['hours'])
    combined_df = combined_df.drop_duplicates()
    combined_datas.append(combined_df)

combined_datas_df = pd.concat(combined_datas, ignore_index=True)




In [ ]:
combined_datas_df_filter = combined_datas_df[combined_datas_df['showed_hours_today'] >= 12].reset_index(drop=True)
total_hours_2021 = combined_datas_df_filter.groupby(['device_aid','latitude', 'longitude']).agg({'duration_location_hour':'sum'}).reset_index()

In [ ]:
total_hours_with_home, home_location_2021 = find_and_save_primary_location(
    df=total_hours_2021,
    id_col='device_aid',
    duration_col='duration_location_hour',
    location_cols=['device_aid', 'latitude', 'longitude'],
    location_type_name='home',
    output_path="/nas/houce/UK_mobility/data_20251226/home_workplace_location/home_location_2021.csv"
)

## Step 2: showed up days selection

In [ ]:
combined_datas_df_filter_2021 = combined_datas_df[combined_datas_df['showed_hours_today'] >= 12].reset_index(drop=True)
combined_datas_df_uni_2021 = combined_datas_df_filter_2021.groupby(['device_aid','day']).size()
combined_datas_df_uni_2021 = combined_datas_df_uni_2021.reset_index().drop(columns=0)
# combined_datas_df_uni['is_weekend'] = combined_datas_df_uni['day'].apply(lambda x: 1 if pd.to_datetime(x).weekday() >= 5 else 0)
combined_datas_df_uni_2021['is_weekend'] = combined_datas_df_uni_2021['day'].apply(lambda x: 1 if x == 20210306 or x == 20210307 else 0)

In [ ]:
combined_datas_df_2021_week_count = combined_datas_df_uni_2021.groupby(['device_aid','is_weekend']).size().reset_index().rename(columns={0: 'day_count'})

In [ ]:
weekend_count_2021 = combined_datas_df_2021_week_count[combined_datas_df_2021_week_count['is_weekend'] == 1]
weekday_count_2021 = combined_datas_df_2021_week_count[combined_datas_df_2021_week_count['is_weekend'] == 0]
weekday_count_2021_all = weekday_count_2021.merge(weekend_count_2021, on='device_aid', how='outer', suffixes=('_weekday', '_weekend'))
weekday_count_2021_all = weekday_count_2021_all.drop(columns=['is_weekend_weekday', 'is_weekend_weekend'])
weekday_count_2021_all = weekday_count_2021_all.fillna(0)
weekday_count_2021_all['day_count_all'] = weekday_count_2021_all['day_count_weekday'] + weekday_count_2021_all['day_count_weekend']
weekday_count_2021_all.to_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/weekday_weekend_count_2021.csv", index=False)

In [ ]:
home_data_list = glob("/nas/houce/UK_mobility/data_20251226/home/home_UK_2020_*.csv")
home_data_list.sort(key=lambda x: os.path.basename(x).split('_')[1] + os.path.basename(x).split('_')[2])

workplace_data_list = glob("/nas/houce/UK_mobility/data_20251226/workplace/workplace_UK_2020_*.csv")
workplace_data_list.sort(key=lambda x: os.path.basename(x).split('_')[1] + os.path.basename(x).split('_')[2])

In [ ]:
from tqdm import tqdm
combined_datas_2020 = []
for i, file_path in tqdm(enumerate(home_data_list), total=len(home_data_list)):
    home_df = pd.read_csv(file_path)
    workplace_df = pd.read_csv(workplace_data_list[i])
    home_df['workplace'] = 0
    combined_df = pd.concat([home_df, workplace_df], ignore_index=True)
    combined_datas_2020.append(combined_df)

combined_datas_df_2020 = pd.concat(combined_datas_2020, ignore_index=True)

In [ ]:
combined_datas_df_filter_2020 = combined_datas_df_2020[combined_datas_df_2020['showed_hours_today'] >= 12].reset_index(drop=True)
total_hours_2020 = combined_datas_df_filter_2020.groupby(['device_aid','latitude', 'longitude']).agg({'duration_location_hour':'sum'}).reset_index()

combined_datas_df_uni_2020 = combined_datas_df_filter_2020.groupby(['device_aid','day']).size()
combined_datas_df_uni_2020 = combined_datas_df_uni_2020.reset_index().drop(columns=0)
# combined_datas_df_uni['is_weekend'] = combined_datas_df_uni['day'].apply(lambda x: 1 if pd.to_datetime(x).weekday() >= 5 else 0)
combined_datas_df_uni_2020['is_weekend'] = combined_datas_df_uni_2020['day'].apply(lambda x: 1 if x == 20200201 or x == 20200202 else 0)

In [ ]:
combined_datas_df_2020_week_count = combined_datas_df_uni_2020.groupby(['device_aid','is_weekend']).size().reset_index().rename(columns={0: 'day_count'})

In [ ]:
weekend_count_2020 = combined_datas_df_2020_week_count[combined_datas_df_2020_week_count['is_weekend'] == 1]
weekday_count_2020 = combined_datas_df_2020_week_count[combined_datas_df_2020_week_count['is_weekend'] == 0]
weekday_count_2020_all = weekday_count_2020.merge(weekend_count_2020, on='device_aid', how='outer', suffixes=('_weekday', '_weekend'))
weekday_count_2020_all = weekday_count_2020_all.drop(columns=['is_weekend_weekday', 'is_weekend_weekend'])
weekday_count_2020_all = weekday_count_2020_all.fillna(0)
weekday_count_2020_all['day_count_all'] = weekday_count_2020_all['day_count_weekday'] + weekday_count_2020_all['day_count_weekend']
weekday_count_2020_all.to_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/weekday_weekend_count_2020.csv", index=False)

In [ ]:
# weekday_count_2021_all = pd.read_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/weekday_weekend_count_2021.csv")

In [ ]:
weekday_count_2020_2021 = weekday_count_2020_all.merge(weekday_count_2021_all, on='device_aid', how='outer', suffixes=('_2020', '_2021'))
weekday_count_2020_2021['device_aid_keep'] = weekday_count_2020_2021.apply(lambda row: 1 if (row['day_count_all_2020'] >= 4) & (row['day_count_all_2021'] >= 4) else 0, axis=1)
# weekday_count_2020_2021.to_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/weekday_weekend_count_2020_2021.csv", index=False)
device_aid_keep = list(weekday_count_2020_2021[weekday_count_2020_2021['device_aid_keep'] == 1]['device_aid'])

In [ ]:
combined_datas_df_2020_select = combined_datas_df_filter_2020[combined_datas_df_filter_2020['device_aid'].isin(device_aid_keep)].drop(columns=['home', 'workplace'])
combined_datas_df_2021_select = combined_datas_df_filter_2021[combined_datas_df_filter_2021['device_aid'].isin(device_aid_keep)].drop(columns=['home', 'workplace'])
combined_datas_df_2020_select['weighted_hours_today'] = combined_datas_df_2020_select['duration_location_hour'] / combined_datas_df_2020_select['showed_hours_today'] * 24
combined_datas_df_2021_select['weighted_hours_today'] = combined_datas_df_2021_select['duration_location_hour'] / combined_datas_df_2021_select['showed_hours_today'] * 24


In [ ]:
home_location_2020 = pd.read_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/home_location_2020.csv")
home_location_2021 = pd.read_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/home_location_2021.csv")
workplace_location = pd.read_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/workplace_location.csv")

home_location_2020['home'] = 1
home_location_2021['home'] = 1
workplace_location['workplace'] = 1

combined_datas_df_2020_select_merge = combined_datas_df_2020_select.merge(home_location_2020, on=['device_aid','latitude','longitude'], how='left')
combined_datas_df_2021_select_merge = combined_datas_df_2021_select.merge(home_location_2021, on=['device_aid','latitude','longitude'], how='left')

combined_datas_df_2020_select_merge_workplace = combined_datas_df_2020_select_merge.merge(workplace_location, on=['device_aid','latitude','longitude'], how='left')
combined_datas_df_2021_select_merge_workplace = combined_datas_df_2021_select_merge.merge(workplace_location, on=['device_aid','latitude','longitude'], how='left')

home_2020 = combined_datas_df_2020_select_merge_workplace[combined_datas_df_2020_select_merge_workplace['home'] == 1]
workplace_2020 = combined_datas_df_2020_select_merge_workplace[combined_datas_df_2020_select_merge_workplace['workplace'] == 1]
home_2021 = combined_datas_df_2021_select_merge_workplace[combined_datas_df_2021_select_merge_workplace['home'] == 1]
workplace_2021 = combined_datas_df_2021_select_merge_workplace[combined_datas_df_2021_select_merge_workplace['workplace'] == 1]
amenity_2020 = combined_datas_df_2020_select_merge_workplace[(combined_datas_df_2020_select_merge_workplace['home'] != 1) & (combined_datas_df_2020_select_merge_workplace['workplace'] != 1)]
amenity_2021 = combined_datas_df_2021_select_merge_workplace[(combined_datas_df_2021_select_merge_workplace['home'] != 1) & (combined_datas_df_2021_select_merge_workplace['workplace'] != 1)]

home_mean_2020 = home_2020.groupby('device_aid').agg({'weighted_hours_today':'mean'}).reset_index().rename(columns={'weighted_hours_today':'home_weighted_hours_today'})
home_mean_2021 = home_2021.groupby('device_aid').agg({'weighted_hours_today':'mean'}).reset_index().rename(columns={'weighted_hours_today':'home_weighted_hours_today'})

workplace_mean_2020 = workplace_2020.groupby('device_aid').agg({'weighted_hours_today':'mean'}).reset_index().rename(columns={'weighted_hours_today':'home_weighted_hours_today'})
workplace_mean_2021 = workplace_2021.groupby('device_aid').agg({'weighted_hours_today':'mean'}).reset_index().rename(columns={'weighted_hours_today':'home_weighted_hours_today'})

In [ ]:
home_2020_freq = home_2020[['device_aid','latitude','longitude','day','weighted_hours_today', 'showed_hours_today']]
home_2020_freq['place'] = 'home'
home_2021_freq = home_2021[['device_aid','latitude','longitude','day','weighted_hours_today', 'showed_hours_today']]
home_2021_freq['place'] = 'home'

workplace_2020_freq = workplace_2020[['device_aid','latitude','longitude','day','weighted_hours_today', 'showed_hours_today']]
workplace_2020_freq['place'] = 'workplace'
workplace_2021_freq = workplace_2021[['device_aid','latitude','longitude','day','weighted_hours_today', 'showed_hours_today']]
workplace_2021_freq['place'] = 'workplace'

amenity_2020_freq = amenity_2020[['device_aid','latitude','longitude','day','weighted_hours_today', 'showed_hours_today']]
amenity_2020_freq['place'] = 'amenity'
amenity_2021_freq = amenity_2021[['device_aid','latitude','longitude','day','weighted_hours_today', 'showed_hours_today']]
amenity_2021_freq['place'] = 'amenity'



In [ ]:
total_2020 = pd.concat([home_2020_freq, workplace_2020_freq, amenity_2020_freq], ignore_index=True)
total_2021 = pd.concat([home_2021_freq, workplace_2021_freq, amenity_2021_freq], ignore_index=True)

total_2020.to_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/total_2020_selected.csv", index=False)
total_2021.to_csv("/nas/houce/UK_mobility/data_20251226/home_workplace_location/total_2021_selected.csv", index=False)

In [ ]:
total_2020 